# Google Colab Intent Classifier

This notebook reproduces the CPU-friendly bidirectional RNN/LSTM intent classifier from `intent_classifier.py`.

In Google Colab, run the upload cell, then run the remaining cells from top to bottom. Locally, the notebook uses `data/intents_train.csv` and `data/intents_test.csv` when they are present.

In [5]:
# Install only the packages that may not already be available in Colab.
%pip install -q torch pandas numpy scikit-learn

import os
import pickle
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
FALLBACK_INTENT = "Fallback Intent"
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z0-9]+)?")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
print(f"PyTorch {torch.__version__}")


ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device: '/home/aatki/.local/lib/python3.10/site-packages/nvidia/cusparselt/LICENSE.txt'

Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'torch'

In [ ]:
# Colab: Upload the missing intents_test.csv file. Locally: this cell uses the repository data/ folder.
from pathlib import Path
import pandas as pd

data_dir = Path("data")
if not (data_dir / "intents_train.csv").exists() or not (data_dir / "intents_test.csv").exists():
    # If not in the data directory, check the current working directory
    if Path("intents_train.csv").exists() and Path("intents_test.csv").exists():
        data_dir = Path(".")
    else:
        try:
            from google.colab import files
            if Path("intents_train.csv").exists():
                print("Found 'intents_train.csv'. Please upload the missing 'intents_test.csv' file.")
            else:
                print("Please upload BOTH intents_train.csv and intents_test.csv")
            uploaded = files.upload()
            data_dir = Path(".")
        except ImportError:
            raise FileNotFoundError("Put intents_train.csv and intents_test.csv in a data/ folder before running.")

train_path = data_dir / "intents_train.csv"
test_path = data_dir / "intents_test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"Could not find training file at {train_path}. Please re-run this cell and upload it.")
if not test_path.exists():
    raise FileNotFoundError(f"Could not find test file at {test_path}. Please re-run this cell and upload it.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

required_columns = {"text", "intent"}
missing = required_columns - set(train_df.columns)
if missing:
    raise ValueError(f"Training data is missing columns: {sorted(missing)}")
if "text" not in test_df.columns:
    raise ValueError("Test data must contain a 'text' column.")

print("Verification Successful!")
print(f"Training dataset: {len(train_df)} rows loaded successfully.")
print(f"Test dataset: {len(test_df)} rows loaded successfully.")
print(f"Intent classes detected: {train_df['intent'].nunique()}")

In [ ]:
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

class Vocabulary:
    def __init__(self, min_frequency=1, max_size=12000):
        self.min_frequency = min_frequency
        self.max_size = max_size
        self.token_to_id = {"<pad>": 0, "<unk>": 1}

    def fit(self, texts):
        counts = {}
        for text in texts:
            for token in tokenize(text):
                counts[token] = counts.get(token, 0) + 1
        ranked = sorted(
            ((count, token) for token, count in counts.items() if count >= self.min_frequency),
            reverse=True,
        )
        for _, token in ranked[: self.max_size - 2]:
            self.token_to_id[token] = len(self.token_to_id)

    def encode(self, text, max_length):
        ids = [self.token_to_id.get(token, 1) for token in tokenize(text)][:max_length]
        return ids + [0] * (max_length - len(ids))

class SequenceClassifier(nn.Module):
    def __init__(self, vocabulary_size, class_count, architecture, embedding_dim=96, hidden_dim=96):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=0)
        recurrent = nn.RNN if architecture == "rnn" else nn.LSTM
        self.recurrent = recurrent(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.25)
        self.output = nn.Linear(hidden_dim * 2, class_count)

    def forward(self, tokens):
        _, state = self.recurrent(self.embedding(tokens))
        hidden = state[0] if isinstance(state, tuple) else state
        pooled = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.output(self.dropout(pooled))

def train_architecture(train_texts, train_labels, validation_texts, validation_labels,
                       vocabulary, architecture, max_length, epochs, batch_size, class_count):
    model = SequenceClassifier(len(vocabulary.token_to_id), class_count, architecture)
    train_x = torch.tensor([vocabulary.encode(text, max_length) for text in train_texts], dtype=torch.long)
    train_y = torch.tensor(train_labels, dtype=torch.long)
    validation_x = torch.tensor([vocabulary.encode(text, max_length) for text in validation_texts], dtype=torch.long)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size, shuffle=True)
    best_accuracy = -1.0
    best_state = None
    for epoch in range(epochs):
        model.train()
        for tokens, targets in loader:
            optimizer.zero_grad()
            loss = loss_function(model(tokens), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            predictions = model(validation_x).argmax(dim=1).numpy()
        current_accuracy = accuracy_score(validation_labels, predictions)
        if current_accuracy >= best_accuracy:
            best_accuracy = current_accuracy
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"{architecture.upper()} epoch {epoch + 1}/{epochs}: {current_accuracy:.3f}")
    model.load_state_dict(best_state)
    return model, best_accuracy

In [ ]:
# Train both architectures and keep the one with the best validation accuracy.
seed_everything()
labels = sorted(train_df["intent"].unique().tolist())
label_ids = {name: index for index, name in enumerate(labels)}
encoded_labels = [label_ids[label] for label in train_df["intent"]]
stratify_labels = train_df["intent"] if train_df["intent"].value_counts().min() >= 2 else None

train_texts, validation_texts, train_labels, validation_labels = train_test_split(
    train_df["text"].tolist(), encoded_labels, test_size=0.2,
    random_state=SEED, stratify=stratify_labels,
)
vocabulary = Vocabulary()
vocabulary.fit(train_texts)
max_length = min(40, max(8, max(len(tokenize(text)) for text in train_df["text"])))

# Reduce epochs here for a quick smoke test; 25 matches the original script.
epochs = 25
batch_size = 32
results = {}
for architecture in ("rnn", "lstm"):
    model, validation_accuracy = train_architecture(
        train_texts, train_labels, validation_texts, validation_labels,
        vocabulary, architecture, max_length, epochs, batch_size, len(labels),
    )
    results[architecture] = (model, validation_accuracy)
    print(f"{architecture.upper()} best validation accuracy: {validation_accuracy:.3f}")

architecture = max(results, key=lambda name: results[name][1])
selected_model, validation_accuracy = results[architecture]
print(f"Selected model: {architecture.upper()}")

In [ ]:
# Calibrate the fallback threshold, classify the validation set, predict on test_df, and save artifacts.
selected_model.eval()

# 1. Validation set evaluation & calibration (using validation_texts)
validation_x = torch.tensor([vocabulary.encode(text, max_length) for text in validation_texts], dtype=torch.long)
with torch.no_grad():
    validation_probabilities = torch.softmax(selected_model(validation_x), dim=1).numpy()

correct_confidences = validation_probabilities[np.arange(len(validation_labels)), validation_labels]
fallback_threshold = float(max(0.45, np.quantile(correct_confidences, 0.05)))
indexes = validation_probabilities.argmax(axis=1)

# 2. Generate final predictions on test_df["text"]
test_x = torch.tensor([vocabulary.encode(text, max_length) for text in test_df["text"]], dtype=torch.long)
with torch.no_grad():
    test_probabilities = torch.softmax(selected_model(test_x), dim=1).numpy()

test_indexes = test_probabilities.argmax(axis=1)
final_predictions = [
    labels[index] if test_probabilities[row, index] >= fallback_threshold else FALLBACK_INTENT
    for row, index in enumerate(test_indexes)
]

# Save test predictions
output_dir = Path(".")
pd.DataFrame({"intent": final_predictions}).to_csv(output_dir / "intents.csv", index=False)

# Save model package
model_bundle = {
    "model_state_dict": selected_model.state_dict(),
    "architecture": architecture,
    "labels": labels,
    "vocabulary": vocabulary.token_to_id,
    "max_length": max_length,
    "fallback_threshold": fallback_threshold,
}
with open(output_dir / "model.pkl", "wb") as handle:
    pickle.dump(model_bundle, handle)

print(f"Validation accuracy: {validation_accuracy:.3f}")
print(f"Fallback threshold: {fallback_threshold:.3f}")
print("\n=== Classification Report on Validation Set ===")
print(classification_report(
    validation_labels, indexes, labels=list(range(len(labels))),
    target_names=labels, zero_division=0,
))
print(f"Successfully wrote {len(final_predictions)} predictions using  to intents.csv!")

## Optional: BERT transfer-learning experiment

This section downloads a pretrained DistilBERT model from Hugging Face. Enable a Colab GPU for faster training. It is separate from the default RNN/LSTM pipeline because pretrained-weight downloads require internet access.

In [ ]:
# Install the optional BERT dependencies only when you want to run this experiment.
%pip install -q transformers accelerate

from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

bert_model_name = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=len(labels),
    id2label={index: label for index, label in enumerate(labels)},
    label2id=label_ids,
)
bert_train_dataset = IntentDataset(train_texts, train_labels, bert_tokenizer)
bert_validation_dataset = IntentDataset(validation_texts, validation_labels, bert_tokenizer)

def compute_metrics(eval_prediction):
    logits, targets = eval_prediction
    return {"accuracy": accuracy_score(targets, np.argmax(logits, axis=-1))}

bert_args = TrainingArguments(
    output_dir="bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    load_best_model_at_end=False,
)
bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_train_dataset,
    eval_dataset=bert_validation_dataset,
    compute_metrics=compute_metrics,
)
bert_trainer.train()
print(bert_trainer.evaluate())

In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score

# Retrieve the best validation accuracies
rnn_accuracy = results['rnn'][1]
lstm_accuracy = results['lstm'][1]

# Evaluate the BERT model to get its validation accuracy
bert_eval_results = bert_trainer.evaluate()
bert_accuracy = bert_eval_results['eval_accuracy']

print(f"RNN Validation Accuracy: {rnn_accuracy:.3f}")
print(f"LSTM Validation Accuracy: {lstm_accuracy:.3f}")
print(f"BERT Validation Accuracy: {bert_accuracy:.3f}")

# Compare and pick the absolute best model
model_accuracies = {
    'RNN': rnn_accuracy,
    'LSTM': lstm_accuracy,
    'BERT': bert_accuracy
}

best_model_name = max(model_accuracies, key=model_accuracies.get)
print(f"\nBest overall model: {best_model_name} with {model_accuracies[best_model_name]:.3f} accuracy!")

## Download the generated files in Google Colab

Run the next cell in Colab to download both artifacts to your computer.

In [ ]:
from google.colab import files
files.download("intents.csv")
files.download("model.pkl")
